# Chapter 10: Modern RL Algorithms: GRPO, RLOO, KTO & More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part2_core/10_grpo_rloo_kto.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 10:** Modern RL Algorithms: GRPO, RLOO, KTO & More  
> **Notebook:** `part2_core/10_grpo_rloo_kto.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 10 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q transformers==4.40.0 trl==0.8.6 accelerate==0.29.3

print('Environment ready.')

In [ ]:
import re
import copy
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Shared Setup — Model & Verifiable Reward

We use `Qwen/Qwen2.5-0.5B` (82 M params) and simple arithmetic questions. The reward is **verifiable**: 1.0 if the model's output contains the correct numeric answer, 0.0 otherwise.

This mirrors the setup used by DeepSeek-R1 and Qwen-2.5-Math at much larger scale.

In [ ]:
MATH_QUESTIONS = [
    {"question": "What is 2 + 2?",       "answer": "4"},
    {"question": "What is 5 * 3?",       "answer": "15"},
    {"question": "What is 10 - 4?",      "answer": "6"},
    {"question": "What is 8 + 7?",       "answer": "15"},
    {"question": "What is 6 * 6?",       "answer": "36"},
    {"question": "What is 20 - 8?",      "answer": "12"},
    {"question": "What is 3 + 9?",       "answer": "12"},
    {"question": "What is 7 * 4?",       "answer": "28"},
    {"question": "What is 100 - 45?",    "answer": "55"},
    {"question": "What is 9 + 6?",       "answer": "15"},
    {"question": "What is 4 * 8?",       "answer": "32"},
    {"question": "What is 50 - 17?",     "answer": "33"},
    {"question": "What is 11 + 13?",     "answer": "24"},
    {"question": "What is 5 * 5?",       "answer": "25"},
    {"question": "What is 30 - 14?",     "answer": "16"},
]


def verifiable_reward(response: str, correct_answer: str) -> float:
    """
    Returns 1.0 if `correct_answer` appears as a standalone number in `response`,
    else 0.0.  Uses word-boundary matching so '15' does not match '150'.
    """
    pattern = r'(?<![\d])' + re.escape(correct_answer) + r'(?![\d])'
    return 1.0 if re.search(pattern, response) else 0.0


# Quick sanity checks
print(verifiable_reward('The answer is 15.',  '15'))   # 1.0
print(verifiable_reward('It is 150 I think.', '15'))   # 0.0
print(verifiable_reward('I have no idea.',    '15'))   # 0.0

print(f'Math question pool: {len(MATH_QUESTIONS)}')

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = 'left'


def fresh_model():
    """Load a fresh Qwen/Qwen2.5-0.5B on DEVICE."""
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    return m.to(DEVICE)


def encode_prompt(question: str, max_len: int = 40):
    text = f"Q: {question}\nA:"
    enc  = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    return {k: v.to(DEVICE) for k, v in enc.items()}


def generate_responses(
    model,
    question: str,
    n: int = 8,
    max_new_tokens: int = 20,
):
    """
    Sample `n` independent responses for a question.
    Returns a list of decoded response strings.
    """
    enc = encode_prompt(question)
    prompt_len = enc['input_ids'].shape[1]
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            num_return_sequences=n,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [
        tokenizer.decode(out[i][prompt_len:], skip_special_tokens=True).strip()
        for i in range(n)
    ]


def compute_log_prob(
    model,
    question: str,
    response: str,
    max_len: int = 60,
) -> torch.Tensor:
    """
    Compute the sum of log-probabilities for the response tokens.
    """
    full_text = f"Q: {question}\nA: {response}"
    enc = tokenizer(
        full_text, return_tensors='pt',
        truncation=True, max_length=max_len,
        padding='max_length'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)

    prompt_text = f"Q: {question}\nA:"
    prompt_len  = len(tokenizer.encode(prompt_text))

    logits = model(input_ids=ids, attention_mask=mask).logits
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # (1, T-1, V)
    target    = ids[:, 1:]                                  # (1, T-1)
    tok_lp    = log_probs.gather(2, target.unsqueeze(-1)).squeeze(-1)  # (1, T-1)

    resp_mask = torch.zeros_like(tok_lp)
    resp_mask[:, max(0, prompt_len - 1):] = 1.0
    return (tok_lp * resp_mask).sum(dim=1)  # (1,)


n_params = sum(p.numel() for p in fresh_model().parameters())
print(f'Model: {MODEL_NAME}  |  {n_params/1e6:.1f}M params')

## Algorithm 1 — GRPO (Group Relative Policy Optimisation)

GRPO was introduced by DeepSeek to remove the critic (value network) entirely. Instead of estimating a per-token baseline, it uses the **group mean** as the baseline.

### Algorithm
For each prompt $x$:
1. Sample $G = 8$ responses $\{y_1, \dots, y_G\}$ from the policy $\pi_\theta$.
2. Score each: $r_i = R(x, y_i) \in \{0, 1\}$.
3. Normalise to get advantage estimates:
$$\hat{A}_i = \frac{r_i - \bar{r}}{\text{std}(r) + \epsilon}$$
4. GRPO loss (with KL regularisation against a reference policy $\pi_{\text{ref}}$):
$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^G \hat{A}_i \cdot \log\pi_\theta(y_i|x) + \beta\,\text{KL}[\pi_\theta \| \pi_{\text{ref}}]$$

### Why this works
The group mean is a **valid variance-reducing baseline** — by subtracting it, responses that are better than average within the group get a positive advantage, and worse-than-average responses get a negative advantage.

In [ ]:
def grpo_loss(
    policy: nn.Module,
    ref_policy: nn.Module,
    question: str,
    responses: list,
    rewards: list,
    beta: float = 0.04,
) -> torch.Tensor:
    """
    GRPO loss for one question with G responses.

    Parameters
    ----------
    policy      : current policy model (requires grad)
    ref_policy  : frozen reference policy
    question    : prompt string
    responses   : list of G response strings
    rewards     : list of G float rewards
    beta        : KL penalty coefficient

    Returns
    -------
    scalar loss
    """
    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)

    # Normalise rewards to advantages
    r_mean = rewards_t.mean()
    r_std  = rewards_t.std() + 1e-8
    advantages = (rewards_t - r_mean) / r_std  # (G,)

    total_loss = torch.tensor(0.0, device=DEVICE, requires_grad=False)

    policy_log_probs = []
    ref_log_probs    = []

    for resp in responses:
        lp_policy = compute_log_prob(policy,     question, resp)
        with torch.no_grad():
            lp_ref = compute_log_prob(ref_policy, question, resp)
        policy_log_probs.append(lp_policy)
        ref_log_probs.append(lp_ref)

    policy_lps = torch.cat(policy_log_probs)   # (G,)
    ref_lps    = torch.cat(ref_log_probs)       # (G,)

    # KL: E[log(pi_policy) - log(pi_ref)]
    kl = (policy_lps - ref_lps).mean()

    # Policy gradient term
    pg_loss = -(advantages * policy_lps).mean()

    return pg_loss + beta * kl


print('GRPO loss function defined.')

In [ ]:
G = 8             # group size
GRPO_STEPS = 50
BETA_GRPO  = 0.04

grpo_policy = fresh_model()
grpo_ref    = copy.deepcopy(grpo_policy)
grpo_ref.eval()
for p in grpo_ref.parameters():
    p.requires_grad = False

grpo_opt = AdamW(grpo_policy.parameters(), lr=5e-6)

grpo_reward_log    = []   # mean reward per step
grpo_accuracy_log  = []   # fraction of correct responses per step
grpo_variance_log  = []   # variance of rewards within group (for later comparison)

print(f'Training GRPO for {GRPO_STEPS} steps (G={G}) ...')

for step in range(GRPO_STEPS):
    qa = random.choice(MATH_QUESTIONS)
    question = qa['question']
    answer   = qa['answer']

    # 1. Generate G responses
    responses = generate_responses(grpo_policy, question, n=G)

    # 2. Score each
    rewards = [verifiable_reward(r, answer) for r in responses]

    # 3-4. GRPO loss
    grpo_policy.train()
    loss = grpo_loss(grpo_policy, grpo_ref, question, responses, rewards, beta=BETA_GRPO)

    # 5. Update
    grpo_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(grpo_policy.parameters(), 1.0)
    grpo_opt.step()

    mean_r = float(np.mean(rewards))
    var_r  = float(np.var(rewards))
    acc    = float(np.mean([r > 0 for r in rewards]))
    grpo_reward_log.append(mean_r)
    grpo_accuracy_log.append(acc)
    grpo_variance_log.append(var_r)

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}/{GRPO_STEPS}  loss={loss.item():.4f}  '
              f'mean_r={mean_r:.3f}  acc={acc:.2%}')

print('GRPO training complete!')

## Algorithm 2 — RLOO (REINFORCE Leave-One-Out)

RLOO is a simple but powerful variant: instead of subtracting the **group mean**, each response subtracts the mean of all **other** responses in the group. This gives an unbiased, lower-variance baseline.

### Advantage estimate

$$\hat{A}_i^{\text{RLOO}} = r_i - \frac{1}{G-1}\sum_{j \neq i} r_j$$

When $G = 2$ this reduces to $r_1 - r_2$ and $r_2 - r_1$ — exactly the DPO margin.

### GRPO vs. RLOO variance
RLOO uses a tighter baseline (excluding the own sample), which **reduces estimator variance** compared to GRPO's group mean. We measure this empirically below.

In [ ]:
def rloo_advantages(rewards: list) -> torch.Tensor:
    """
    Leave-one-out baseline advantage estimates.

    A_i = r_i - mean(r_j for j != i)

    Parameters
    ----------
    rewards : list of G float rewards

    Returns
    -------
    advantages : (G,) tensor
    """
    G = len(rewards)
    r = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
    total = r.sum()
    # leave-one-out mean = (total - r_i) / (G - 1)
    loo_mean = (total - r) / max(G - 1, 1)
    return r - loo_mean   # (G,)


def rloo_loss(
    policy: nn.Module,
    ref_policy: nn.Module,
    question: str,
    responses: list,
    rewards: list,
    beta: float = 0.04,
) -> torch.Tensor:
    """
    RLOO policy gradient loss with KL regularisation.
    Same interface as grpo_loss but uses leave-one-out advantages.
    """
    advantages = rloo_advantages(rewards)   # (G,)

    policy_log_probs, ref_log_probs = [], []
    for resp in responses:
        lp_policy = compute_log_prob(policy, question, resp)
        with torch.no_grad():
            lp_ref = compute_log_prob(ref_policy, question, resp)
        policy_log_probs.append(lp_policy)
        ref_log_probs.append(lp_ref)

    policy_lps = torch.cat(policy_log_probs)   # (G,)
    ref_lps    = torch.cat(ref_log_probs)       # (G,)

    kl      = (policy_lps - ref_lps).mean()
    pg_loss = -(advantages * policy_lps).mean()
    return pg_loss + beta * kl


print('RLOO loss function defined.')

In [ ]:
RLOO_STEPS = 50

rloo_policy = fresh_model()
rloo_ref    = copy.deepcopy(rloo_policy)
rloo_ref.eval()
for p in rloo_ref.parameters():
    p.requires_grad = False

rloo_opt = AdamW(rloo_policy.parameters(), lr=5e-6)

rloo_reward_log    = []
rloo_accuracy_log  = []
rloo_adv_var_log   = []   # variance of advantage estimates

print(f'Training RLOO for {RLOO_STEPS} steps (G={G}) ...')

for step in range(RLOO_STEPS):
    qa = random.choice(MATH_QUESTIONS)
    question = qa['question']
    answer   = qa['answer']

    responses = generate_responses(rloo_policy, question, n=G)
    rewards   = [verifiable_reward(r, answer) for r in responses]

    # Track advantage variance (diagnostic)
    advs = rloo_advantages(rewards)
    rloo_adv_var_log.append(float(advs.var().item()))

    rloo_policy.train()
    loss = rloo_loss(rloo_policy, rloo_ref, question, responses, rewards, beta=BETA_GRPO)

    rloo_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rloo_policy.parameters(), 1.0)
    rloo_opt.step()

    mean_r = float(np.mean(rewards))
    acc    = float(np.mean([r > 0 for r in rewards]))
    rloo_reward_log.append(mean_r)
    rloo_accuracy_log.append(acc)

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}/{RLOO_STEPS}  loss={loss.item():.4f}  '
              f'mean_r={mean_r:.3f}  acc={acc:.2%}')

print('RLOO training complete!')

In [ ]:
# Compare advantage variance: GRPO vs RLOO
# We re-compute GRPO advantage variance from stored reward variance
grpo_adv_var_log = []
for i in range(GRPO_STEPS):
    # Re-sample variance proxy: same reward variance produces same normalised adv variance of 1
    # (GRPO uses z-score normalisation — variance of advantages is always 1 by construction)
    grpo_adv_var_log.append(1.0)  # by construction after z-score

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

steps = np.arange(1, max(GRPO_STEPS, RLOO_STEPS) + 1)

# Accuracy comparison
def smooth(data, w=7):
    if len(data) < w:
        return np.array(data)
    return np.convolve(data, np.ones(w)/w, mode='valid')

sg = smooth(grpo_accuracy_log)
sr = smooth(rloo_accuracy_log)
axes[0].plot(grpo_accuracy_log, alpha=0.2, color='steelblue')
axes[0].plot(np.arange(1, len(sg)+1), sg, color='steelblue', linewidth=2, label='GRPO')
axes[0].plot(rloo_accuracy_log, alpha=0.2, color='orange')
axes[0].plot(np.arange(1, len(sr)+1), sr, color='orange', linewidth=2, label='RLOO')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Fraction Correct (within group)')
axes[0].set_title('GRPO vs RLOO — Answer Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Advantage variance comparison
# RLOO variance is stochastic, GRPO is normalised to 1.0
axes[1].axhline(1.0, color='steelblue', linewidth=2, linestyle='--', label='GRPO (z-score, always 1.0)')
axes[1].plot(rloo_adv_var_log, alpha=0.3, color='orange')
axes[1].plot(smooth(rloo_adv_var_log), color='orange', linewidth=2, label='RLOO (raw variance)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Advantage Variance')
axes[1].set_title('GRPO vs RLOO — Advantage Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grpo_vs_rloo.png', dpi=100, bbox_inches='tight')
plt.show()

## Algorithm 3 — KTO (Kahneman-Tversky Optimisation)

KTO (Ethayarajh et al., 2024) takes inspiration from **prospect theory**: humans are more sensitive to losses than equivalent gains. This motivates an asymmetric loss function that works with **individual thumbs-up / thumbs-down labels** — no pairwise comparisons needed.

### KTO loss

Given a response $y$ with label $z \in \{+1, -1\}$ (1 = desirable, -1 = undesirable):

$$\hat{r}_\theta(x, y) = \beta\left(\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} - z_{\text{KL}}\right)$$

where $z_{\text{KL}}$ is the KL penalty term (computed from the batch).

$$\mathcal{L}_{\text{KTO}} = \begin{cases}\lambda_D\left(1 - \sigma(\hat{r}_\theta)\right) & \text{if } z = +1 \\ \lambda_U\left(1 - \sigma(-\hat{r}_\theta)\right) & \text{if } z = -1\end{cases}$$

The asymmetric weights $\lambda_D$ (desirable) and $\lambda_U$ (undesirable) embody the loss-aversion of prospect theory.

In [ ]:
def kto_loss(
    policy: nn.Module,
    ref_policy: nn.Module,
    question: str,
    response: str,
    label: int,        # +1 = desirable, -1 = undesirable
    beta: float = 0.1,
    lambda_d: float = 1.0,   # weight for desirable responses
    lambda_u: float = 1.0,   # weight for undesirable responses
) -> torch.Tensor:
    """
    KTO loss for a single (question, response, label) triple.

    Parameters
    ----------
    policy     : current policy
    ref_policy : frozen reference
    question   : prompt
    response   : generated response
    label      : +1 (thumbs up) or -1 (thumbs down)
    beta       : temperature scaling
    lambda_d   : desirable loss weight
    lambda_u   : undesirable loss weight

    Returns
    -------
    scalar loss
    """
    lp_policy = compute_log_prob(policy, question, response)          # (1,)
    with torch.no_grad():
        lp_ref = compute_log_prob(ref_policy, question, response)     # (1,)

    # KL term approximated by a single sample
    kl = (lp_policy - lp_ref).detach()                                # (1,)

    # Implicit reward
    r_hat = beta * (lp_policy - lp_ref - kl)                         # (1,)

    if label == 1:    # desirable
        loss = lambda_d * (1.0 - torch.sigmoid(r_hat))
    else:             # undesirable
        loss = lambda_u * (1.0 - torch.sigmoid(-r_hat))

    return loss.mean()


print('KTO loss function defined.')

In [ ]:
KTO_STEPS = 50

kto_policy = fresh_model()
kto_ref    = copy.deepcopy(kto_policy)
kto_ref.eval()
for p in kto_ref.parameters():
    p.requires_grad = False

kto_opt = AdamW(kto_policy.parameters(), lr=5e-6)

kto_reward_log   = []
kto_accuracy_log = []
kto_label_log    = []   # track which label was used

print(f'Training KTO for {KTO_STEPS} steps ...')

for step in range(KTO_STEPS):
    qa = random.choice(MATH_QUESTIONS)
    question = qa['question']
    answer   = qa['answer']

    # Generate ONE response
    responses = generate_responses(kto_policy, question, n=1)
    response  = responses[0]

    # Binary label: thumbs-up if correct, thumbs-down otherwise
    reward = verifiable_reward(response, answer)
    label  = 1 if reward > 0 else -1

    kto_policy.train()
    loss = kto_loss(
        kto_policy, kto_ref,
        question, response, label,
        beta=0.1, lambda_d=1.0, lambda_u=1.0
    )

    kto_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(kto_policy.parameters(), 1.0)
    kto_opt.step()

    kto_reward_log.append(reward)
    kto_accuracy_log.append(float(reward > 0))
    kto_label_log.append(label)

    if (step + 1) % 10 == 0:
        recent_acc = float(np.mean(kto_accuracy_log[-10:]))
        print(f'Step {step+1:3d}/{KTO_STEPS}  loss={loss.item():.4f}  '
              f'recent_acc={recent_acc:.2%}')

print('KTO training complete!')

## Comparing GRPO, RLOO, and KTO

We plot reward accuracy over training steps for all three algorithms and produce a final summary comparison table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps = np.arange(1, GRPO_STEPS + 1)

# Running accuracy (smoothed)
sg = smooth(grpo_accuracy_log)
sr = smooth(rloo_accuracy_log)
sk = smooth(kto_accuracy_log)

axes[0].plot(grpo_accuracy_log, alpha=0.15, color='steelblue')
axes[0].plot(np.arange(1, len(sg)+1), sg, color='steelblue', linewidth=2, label='GRPO')
axes[0].plot(rloo_accuracy_log, alpha=0.15, color='orange')
axes[0].plot(np.arange(1, len(sr)+1), sr, color='orange', linewidth=2, label='RLOO')
axes[0].plot(kto_accuracy_log, alpha=0.15, color='green')
axes[0].plot(np.arange(1, len(sk)+1), sk, color='green', linewidth=2, label='KTO')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Accuracy (correct answers)')
axes[0].set_title('GRPO vs RLOO vs KTO — Answer Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reward comparison (mean over last 20 steps)
algorithms = ['GRPO', 'RLOO', 'KTO']
last20_means = [
    float(np.mean(grpo_accuracy_log[-20:])),
    float(np.mean(rloo_accuracy_log[-20:])),
    float(np.mean(kto_accuracy_log[-20:])),
]
colors = ['steelblue', 'orange', 'green']
bars = axes[1].bar(algorithms, last20_means, color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, last20_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2%}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Mean Accuracy (last 20 steps)')
axes[1].set_title('Final Performance Comparison')
axes[1].set_ylim([0, 1.15])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('algo_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## GRPO Reward over 50 Steps — Zoomed View

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(grpo_reward_log, alpha=0.25, color='steelblue', label='raw mean reward')
ax.plot(np.arange(1, len(smooth(grpo_reward_log))+1),
        smooth(grpo_reward_log), color='steelblue', linewidth=2, label='smoothed')

# Highlight steps where at least one correct answer in group
correct_steps = [i+1 for i, r in enumerate(grpo_reward_log) if r > 0]
if correct_steps:
    ax.scatter(correct_steps,
               [grpo_reward_log[i-1] for i in correct_steps],
               color='gold', zorder=5, s=40, label='any correct in group')

ax.set_xlabel('GRPO Step')
ax.set_ylabel('Mean Reward (fraction correct in group)')
ax.set_title('GRPO: Group Mean Reward over Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grpo_reward.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'GRPO  early (steps 1-10)  avg accuracy: {np.mean(grpo_accuracy_log[:10]):.2%}')
print(f'GRPO  late  (steps 41-50) avg accuracy: {np.mean(grpo_accuracy_log[40:]):.2%}')

## KTO — Prospect Theory Visualisation

The KTO loss surface shows the asymmetric sensitivity to good vs. bad responses — a direct encoding of Kahneman & Tversky's value function.

In [ ]:
r_hat_vals = torch.linspace(-4, 4, 200)

loss_desirable   = 1.0 - torch.sigmoid(r_hat_vals)    # lambda_D * (1 - sigma(r_hat))
loss_undesirable = 1.0 - torch.sigmoid(-r_hat_vals)   # lambda_U * (1 - sigma(-r_hat))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r_hat_vals.numpy(), loss_desirable.numpy(),
        color='steelblue', linewidth=2, label='Desirable (thumbs-up)')
ax.plot(r_hat_vals.numpy(), loss_undesirable.numpy(),
        color='crimson', linewidth=2, linestyle='--', label='Undesirable (thumbs-down)')
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Implicit reward r_hat = β(log π/π_ref - KL)')
ax.set_ylabel('KTO Loss')
ax.set_title('KTO Loss Surface — Prospect Theory Asymmetry')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('kto_loss_surface.png', dpi=100, bbox_inches='tight')
plt.show()

print('Desirable responses: loss decreases as r_hat increases (model should generate more of this).')
print('Undesirable responses: loss decreases as r_hat decreases (model should avoid this).')

## Algorithm Comparison Table

| Algorithm | Baseline | Data needed | Critic? | Key strength | Key weakness |
|-----------|----------|-------------|---------|-------------|-------------|
| **PPO** | Value network | On-policy rollouts + reward signal | Yes | Stable, well-studied | Requires critic, high memory |
| **GRPO** | Group mean (z-score) | G responses + reward per prompt | No | Simple, no critic | Higher variance than RLOO |
| **RLOO** | Leave-one-out mean | G responses + reward per prompt | No | Lower variance than GRPO | Slightly more compute |
| **KTO** | KL-penalised implicit reward | Single response + binary label | No | No pairs needed, realistic | Less signal per sample |
| **DPO** | Reference log-ratio | Fixed preference pairs | No | No RL loop needed | Off-policy distribution shift |

### When to use each
- **GRPO** / **RLOO**: When you have a verifiable reward (math, code, factual lookup) and want RL without a critic.
- **KTO**: When preference pairs are expensive but binary thumbs are cheap (e.g., user click-through data).
- **PPO**: When stability is paramount and you can afford the memory overhead of a value network.
- **DPO**: When you have a large, high-quality static preference dataset and no online RL budget.

> **Next chapter preview** → Part 3 of the book covers advanced topics: Constitutional AI, process reward models, scalable oversight, and the frontier of alignment research.

In [ ]:
print('=== Chapter 10 — Training Summary ===')
print(f'Model              : {MODEL_NAME} ({n_params/1e6:.1f}M params)')
print(f'Group size G       : {G}')
print(f'Math questions     : {len(MATH_QUESTIONS)}')
print()
print('GRPO (50 steps):')
print(f'  Early accuracy (1-10)  : {np.mean(grpo_accuracy_log[:10]):.2%}')
print(f'  Late  accuracy (41-50) : {np.mean(grpo_accuracy_log[40:]):.2%}')
print()
print('RLOO (50 steps):')
print(f'  Early accuracy (1-10)  : {np.mean(rloo_accuracy_log[:10]):.2%}')
print(f'  Late  accuracy (41-50) : {np.mean(rloo_accuracy_log[40:]):.2%}')
print()
print('KTO (50 steps):')
print(f'  Thumbs-up labels       : {kto_label_log.count(1)}/{KTO_STEPS}')
print(f'  Thumbs-down labels     : {kto_label_log.count(-1)}/{KTO_STEPS}')
print(f'  Early accuracy (1-10)  : {np.mean(kto_accuracy_log[:10]):.2%}')
print(f'  Late  accuracy (41-50) : {np.mean(kto_accuracy_log[40:]):.2%}')